# Observability Introduction: Start the DOG Stack

**Module 1: Observability Foundations in the DOG Stack** · [Lesson](course.md) · [Quiz](quiz01_observability_foundations.ipynb)

In this Lab you start the DOG stack (Doris, the OpenTelemetry Collector, and Grafana) and the OpenTelemetry demo, an online store whose services keep sending telemetry to the stack. You check in Doris that the telemetry arrives, and you look at the store and at Grafana. Then you start the incident that the rest of the course investigates: with one of the demo's feature flags, you make half of the store's payments fail, and you watch the failed orders appear in Doris.

Plan about 35 minutes, plus a few minutes for the first download of the images (about 4.8 GB). Run the cells in order.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Start the DOG stack

The environment cell checks out a pinned version of the DOG stack under `.runtime/` and starts its four containers: `doris-obs` (Doris), `doris-obs-otel-collector`, `doris-obs-grafana`, and `doris-obs-mailpit`, a local mail server that receives Grafana's alert emails in Module 8. It also mounts the Doris FE's log directory on this machine, at `.runtime/doris-logs/fe/`, so that Module 2 can load the FE log.

In [ ]:
lab.prepare_environment()


**Expected result**

The host port check lists seven ports as `available`, or as `used by this course` when you rerun the cell. The command log ends with the four running containers. The Doris nodes table shows one FE and one BE, both alive, with the version `doris-4.1.3-rc02-7126cf65d96`, and the last card gives the Grafana, Doris, and Mailpit addresses. The first run downloads about 2.1 GB of images; a rerun reuses the running containers and takes a few seconds.

## 2. Start the OpenTelemetry demo

The OpenTelemetry demo is an online store, the Astronomy Shop, built from more than a dozen services that are instrumented with OpenTelemetry. Its load generator keeps browsing the store and placing orders, so the services keep producing traces, logs, and metrics. The demo's own Collector forwards them to the DOG Collector, which writes them to Doris. The later modules analyze this data.

The next cell starts the demo's 23 containers. The first run downloads about 2.7 GB of images, which takes a few minutes. The cell then waits until every container is running and the storefront answers.

In [ ]:
lab.start_demo()


**Expected result**

The port check shows `28082` as `available`, or as `used by this course` when you rerun the cell. The command log lists 23 containers named `doris-obs-demo-*`. The cell ends with the card "The demo is running": all 23 containers are running and the storefront answers, usually within half a minute. If a container restarted while the demo started, a table above the card lists it. It is running again, and the next check shows whether its telemetry arrives.

### Check that the telemetry reached Doris

Running containers do not show that any telemetry reached Doris. The DOG Collector writes each signal to its own table in the `otel` database: spans to `otel_traces`, logs to `otel_logs`, and metrics to one table per metric type, such as `otel_metrics_sum` for counters. The next cell counts, for each signal, the rows from the last five minutes and the services that sent them. The tables store timestamps in UTC, so the query compares them with `UTC_TIMESTAMP()`. The demo's services start one after another, so the cell reruns the query until all three signals have rows.

In [ ]:
demo_telemetry = lab.wait_for_rows(
    """
    SELECT signal, source_table, services, recent_rows, latest_timestamp
    FROM (
        SELECT 'traces' AS signal, 'otel_traces' AS source_table,
               COUNT(DISTINCT service_name) AS services, COUNT(*) AS recent_rows,
               MAX(timestamp) AS latest_timestamp
        FROM otel_traces
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
        UNION ALL
        SELECT 'logs', 'otel_logs', COUNT(DISTINCT service_name), COUNT(*), MAX(timestamp)
        FROM otel_logs
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
        UNION ALL
        SELECT 'metrics', 'otel_metrics_sum', COUNT(DISTINCT service_name), COUNT(*), MAX(timestamp)
        FROM otel_metrics_sum
        WHERE timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
    ) AS recent
    WHERE recent_rows > 0
    ORDER BY signal
    """,
    title="Telemetry from the last five minutes",
    hint="Read the demo Collector's log with docker logs doris-obs-demo-otel-collector and the DOG Collector's "
         "log with docker logs doris-obs-otel-collector, then rerun this cell.",
    min_rows=3,
    timeout_seconds=180,
)

**Expected result**

Three rows, `logs`, `metrics`, and `traces`, usually within a minute of starting the demo. `services` grows while the demo's services start: after a few minutes, about 18 services send traces (one of them, `frontend-web`, is the storefront code that runs in the browser), about 10 send logs, and about 20 send metrics. The metrics also include the Collectors' own metrics and the host and container metrics, which have an empty service name. `latest_timestamp` is within a few seconds of the current time in UTC.

## 3. Open the storefront

Visit [http://127.0.0.1:28082](http://127.0.0.1:28082) (or the port in `DORIS_OBS_DEMO_PORT`) to see the demo: the Astronomy Shop, an online store for telescopes and accessories. Click **Go Shopping**, open a product, and add it to the cart. Your clicks are requests to the store's services, and they produce telemetry next to the load generator's.

## 4. Open Grafana

Grafana reads the demo's telemetry from Doris. Open [http://127.0.0.1:23000/a/velodb-doris-app/discover](http://127.0.0.1:23000/a/velodb-doris-app/discover) (or the port in `DORIS_OBS_GRAFANA_PORT`); the environment cell in section 1 also prints this address. It opens the **Discover** page of the Doris App, a Grafana plugin for searching the telemetry in Doris, without signing in. The page searches the table `otel_logs`, which the **Table** list at the top names, and shows each matching log record with its columns, such as `timestamp`, `service_name`, and `body`. Depending on your computer's time zone, it may show no records yet: the Doris App filters by your local time, while Doris stores times in UTC. Module 3 explains the page, sets its time range, and finds the failing orders with it.

## 5. Start the incident

The demo can switch on failure scenarios with feature flags, which its flag service, flagd, serves to the other services. This course follows one of them: `paymentFailure` makes the payment service reject a share of its charge requests, so that some customers cannot complete their orders. You turn it on now. The next modules find the failure from the telemetry, as an on-call engineer would, without using what you know about its cause.

1. Open the demo's flag page at [http://127.0.0.1:28082/feature](http://127.0.0.1:28082/feature) (or the port in `DORIS_OBS_DEMO_PORT`). It shows one card for each flag, with a list of the flag's values.
2. In the **paymentFailure** card, "Fail payment service charge requests n%", choose **50%** in the list.
3. Click **save** at the top of the page.

The flag page saves the flags to the file that flagd reads, `doris-opentelemetry-demo/src/flagd/demo.flagd.json` in the checkout under `.runtime/`, and flagd serves the new value at once. The next cell asks flagd for its flags through the storefront's proxy, as the storefront's pages do in your browser, and waits until `paymentFailure` is `50%`.

In [ ]:
lab.wait_for_feature_flag("paymentFailure", "50%");

**Expected result**

The cell first shows its request as a `curl` command, which you can also run in a terminal, then one row: `paymentFailure`, the variant `50%`, and the value `0.5`. The payment service fails a charge when a random number between 0 and 1 is below this value. If you run the cell before you click **save**, it waits for up to three minutes; save the flag, and the cell continues.

### See the failed orders in Doris

An order is one checkout. The checkout service records each order as a span named `oteldemo.CheckoutService/PlaceOrder`, and the span's status is `STATUS_CODE_ERROR` when the order failed. The load generator places only one or two orders a minute, so the next cell reruns its query until the first failed order reaches Doris, for up to ten minutes. While it waits, the status updates every ten seconds with elapsed time, the time remaining, and the number of matching result rows. It turns green when an order is found, or shows the recovery hint when the limit is reached.

In [ ]:
first_failure = lab.wait_for_rows(
    """
    SELECT MIN(timestamp) AS first_failed_order, COUNT(*) AS failed_orders
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 5 MINUTE
    HAVING COUNT(*) > 0
    """,
    title="Failed orders in the last five minutes",
    hint="No order has failed yet. Check that paymentFailure is 50% (the previous cell) and that the demo runs, "
         "then rerun this cell: the load generator places only a few orders a minute.",
    timeout_seconds=600,
)

**Expected result**

One row, usually within two or three minutes of saving the flag: the time of the first failed order of the last five minutes, in UTC, and the number of failed orders so far.

You can also fail an order yourself. In the storefront, add a product to the cart, open the cart, and click **Place Order**. If the order goes through, the store shows "Your order is complete!". If the payment fails, the page stays on the cart and shows no message: the customer only sees that nothing happened. Try again, and about every second order fails.

Count the orders of the last 15 minutes by minute and result:

In [ ]:
lab.sql("""
SELECT DATE_FORMAT(timestamp, '%H:%i') AS minute_utc,
       COUNT(*) AS orders,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_orders
FROM otel_traces
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
GROUP BY minute_utc
ORDER BY minute_utc
""", title="Orders per minute in the last 15 minutes");

**Expected result**

One row for each minute in which an order was placed, usually with 1 to 3 orders. Before you saved the flag, `failed_orders` is 0; since then, about half of the orders fail. With so few orders, a single minute can have none or all of its orders failed: the share evens out over several minutes. This is the symptom that the rest of the course investigates. Module 2 finds the evidence of a failed order in each kind of telemetry, Module 3 finds where the failure starts and why, and Level 2 makes the investigation faster and alerts on it.

## 6. Exercise: find the services that report errors

A failed order leaves error spans in more than one service. Write a query that counts the spans of the last 10 minutes whose status is `STATUS_CODE_ERROR`, by service, with the most errors first.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.sql("""
SELECT service_name, COUNT(*) AS error_spans
FROM otel_traces
WHERE status_code = 'STATUS_CODE_ERROR'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 10 MINUTE
GROUP BY service_name
ORDER BY error_spans DESC
""", title="Error spans of the last 10 minutes, by service");

**Expected result**

The services on the path of an order: `frontend` with about three error spans for each failed order, `frontend-proxy` and `checkout` with about two, and `payment` and `load-generator` with one each. A service records more than one error span for an order when several of its spans for that request end with an error, such as the span of the incoming request and the span of its own call to the next service. An order that you placed in the storefront has no `load-generator` span. A few error spans of other requests may appear as well: right after the demo starts, `frontend` and `payment` fail to reach a cloud metadata service and `accounting` fails to read from Kafka before its topic exists; later, the load generator's browser cancels some image downloads. So error spans show every service that reported the failure, not the service where it started; Module 3 follows one failed order to find that service.

## 7. Stop the environment

The demo runs 23 containers and keeps sending telemetry, which the later modules analyze, so leave it running while you work through the course. When you need the memory back, run `lab.stop_demo()`: the telemetry already in Doris stays, and `lab.start_demo()` starts the demo again. The feature flags stay as you set them, so the payment failure continues after a restart. To end the incident, set **paymentFailure** back to **off** in the flag page.

When you finish the course, `lab.teardown()` removes the course's containers, networks, and volumes, and turns the demo's feature flags off. See Stop and clean up in the [course README](../../README.md).

## You are done when

- the DOG stack's four containers and the demo's 23 containers are running;
- `otel_traces`, `otel_logs`, and `otel_metrics_sum` have rows from the last five minutes;
- `paymentFailure` is `50%`, and Doris has failed orders, spans of `oteldemo.CheckoutService/PlaceOrder` with the status `STATUS_CODE_ERROR`.

## Next

Open [Quiz 1: Observability Foundations in the DOG Stack](quiz01_observability_foundations.ipynb), then continue to [Module 2: Ingesting Observability Data](../module02-data-ingest/course.md).